In [ ]:
!pip install rouge-score 
!pip install --upgrade nltk

In [ ]:
import torch
import os
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from pathlib import Path
import logging
import json
from datetime import datetime
from tqdm import tqdm
from nltk.translate.bleu_score import corpus_bleu
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
import nltk
from typing import Dict, List
from collections import defaultdict
import matplotlib.pyplot as plt
import numpy as np
import os

# Configure basic logging
logging.basicConfig(
    level=print,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

class LightweightCaptioningModel(nn.Module):
    """
    Simplified image captioning model using ResNet18 and GRU decoder.
    More memory-efficient than the mBART version.
    """
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=512):
        super().__init__()
        
        # Use ResNet18 instead of MobileNetV2 for better efficiency
        resnet = models.resnet18(pretrained=True)
        self.image_encoder = nn.Sequential(*list(resnet.children())[:-1])
        
        # Projection for image features - now projects to hidden_dim to match GRU
        self.image_projection = nn.Linear(512, hidden_dim)
        
        # Word embedding layer
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        
        # Additional projection for embedded words to match hidden_dim
        self.embed_projection = nn.Linear(embed_dim, hidden_dim)
        
        # GRU decoder (simpler than Transformer)
        self.decoder = nn.GRU(
            input_size=hidden_dim,  # Changed from embed_dim to hidden_dim
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True
        )
        
        # Output layer
        self.output = nn.Linear(hidden_dim, vocab_size)
        
        # Dropout for regularization
        self.dropout = nn.Dropout(0.3)
        
    def forward(self, images, captions):
        batch_size = images.size(0)
        
        # Encode images
        image_features = self.image_encoder(images)
        image_features = image_features.squeeze(-1).squeeze(-1)
        image_features = self.dropout(self.image_projection(image_features))
        
        # Embed captions and project to hidden_dim
        embedded = self.dropout(self.embedding(captions))
        embedded = self.embed_projection(embedded)
        
        # Initialize hidden state with image features
        hidden = image_features.unsqueeze(0)  # Shape: [1, batch_size, hidden_dim]
        
        # Decode
        output, _ = self.decoder(embedded, hidden)
        output = self.output(output)
        
        return output

class SimpleDataset(Dataset):
    """
    Simplified dataset class focusing on essential functionality.
    """
    def __init__(self, image_dir, captions_file, vocab, max_length=30):
        self.image_dir = Path(image_dir)
        self.vocab = vocab
        self.max_length = max_length
        
        # Basic image transforms
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                              std=[0.229, 0.224, 0.225])
        ])
        
        # Load caption data
        self.samples = []
        
        print("Loading dataset...")
        with open(captions_file, 'r', encoding='utf-8-sig') as f:
            lines = f.readlines()
            for line in tqdm(lines, desc="Loading captions"):
                try:
                    image_name, caption = line.strip().split('\t')
                    image_path = self.image_dir / image_name.split('#')[0]
                    if image_path.exists():
                        self.samples.append((image_name.split('#')[0], caption))
                except:
                    continue
                    
        print(f"Loaded {len(self.samples)} samples")
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        image_name, caption = self.samples[idx]
        
        # Load and transform image
        image = Image.open(self.image_dir / image_name).convert('RGB')
        image = self.transform(image)
        
        # Convert caption to indices
        caption_indices = [self.vocab['<start>']]
        caption_indices.extend(self.vocab.get(token, self.vocab['<unk>']) 
                             for token in caption.split())
        caption_indices.append(self.vocab['<end>'])
        
        # Pad or truncate
        if len(caption_indices) < self.max_length:
            caption_indices.extend([self.vocab['<pad>']] * 
                                 (self.max_length - len(caption_indices)))
        else:
            caption_indices = caption_indices[:self.max_length]
            
        return {
            'image': image,
            'caption': torch.tensor(caption_indices, dtype=torch.long),
            'raw_caption': caption
        }

class UniqueImageDataset(Dataset):
    """
    Dataset class for evaluation, providing unique images with all their reference captions.
    """
    def __init__(self, full_dataset, image_to_captions):
        self.full_dataset = full_dataset
        self.image_to_captions = image_to_captions
        self.image_names = list(image_to_captions.keys())
        # Find one index for each image_name to load the image
        self.image_indices = {}
        for idx, (image_name, _) in enumerate(full_dataset.samples):
            if image_name not in self.image_indices:
                self.image_indices[image_name] = idx

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        image_name = self.image_names[idx]
        # Load image using the full_dataset's method
        image_idx = self.image_indices[image_name]
        sample = self.full_dataset[image_idx]
        image = sample['image']
        captions = self.image_to_captions[image_name]
        return {
            'image': image,
            'captions': captions,
            'image_name': image_name
        }

def create_vocabulary(captions_file, min_freq=1):
    """Create a simple vocabulary from the caption file."""
    word_freq = {}
    
    print("Creating vocabulary...")
    with open(captions_file, 'r', encoding='utf-8-sig') as f:
        lines = f.readlines()
        for line in tqdm(lines, desc="Building vocabulary"):
            try:
                _, caption = line.strip().split('\t')
                for word in caption.split():
                    word_freq[word] = word_freq.get(word, 0) + 1
            except:
                continue
    
    # Create vocabulary
    vocab = {
        '<pad>': 0,
        '<start>': 1,
        '<end>': 2,
        '<unk>': 3
    }
    
    idx = 4
    for word, freq in word_freq.items():
        if freq >= min_freq:
            vocab[word] = idx
            idx += 1
    
    print(f"Created vocabulary with {len(vocab)} tokens")
    return vocab


def train_epoch(model, train_loader, criterion, optimizer, device, epoch, num_epochs):
    """Run one training epoch with progress bar."""
    model.train()
    total_loss = 0
    batch_losses = []  # Track individual batch losses for more detailed plotting
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')
    
    for batch_idx, batch in enumerate(pbar):
        images = batch['image'].to(device)
        captions = batch['caption'].to(device)
        
        outputs = model(images, captions[:, :-1])
        
        loss = criterion(
            outputs.reshape(-1, outputs.shape[-1]),
            captions[:, 1:].reshape(-1)
        )
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        # Save the batch loss
        batch_losses.append(loss.item())
        
        total_loss += loss.item()
        avg_loss = total_loss / (batch_idx + 1)
        pbar.set_postfix({'loss': f'{avg_loss:.4f}'})
        
    epoch_loss = total_loss / len(train_loader)
    return epoch_loss, batch_losses

def evaluate_model(model, descriptions, features, tokenizer, max_length, device):
    """
    Evaluate the image captioning model using the BLEU score methodology
    on the entire dataset rather than a separate test set.
    
    Args:
        model: The trained captioning model
        descriptions: Dictionary mapping image IDs to lists of reference captions
        features: Dictionary mapping image IDs to image features
        tokenizer: The tokenizer used for text processing
        max_length: Maximum caption length
        device: The device to run the model on
    """
    model.eval()
    y, yhat = list(), list()
    
    # Process each image in the dataset
    for key, desc_list in tqdm(descriptions.items(), desc="Evaluating"):
        # Skip if features aren't available for this image
        if key not in features:
            continue
            
        # Generate caption for this image
        image_tensor = features[key].unsqueeze(0).to(device)
        
        # Generate caption
        with torch.no_grad():
            # Encode image
            hidden = model.image_encoder(image_tensor)
            hidden = hidden.squeeze(-1).squeeze(-1)
            hidden = model.image_projection(hidden).unsqueeze(0)
            
            # Start with start token
            decoder_input = torch.tensor([[tokenizer['<start>']]], device=device)
            generated_caption = []
            
            # Generate caption word by word
            for _ in range(max_length):
                embedded = model.embedding(decoder_input)
                embedded = model.embed_projection(embedded)
                output, hidden = model.decoder(embedded, hidden)
                output = model.output(output)
                predicted = output.argmax(2)
                word_idx = predicted[0].item()
                if word_idx == tokenizer['<end>']:
                    break
                generated_caption.append(word_idx)
                decoder_input = predicted
                
        # Convert indices to words
        idx_to_word = {v: k for k, v in tokenizer.items()}
        pred_caption = ['<start>'] + [idx_to_word.get(idx, '<unk>') for idx in generated_caption] + ['<end>']
        
        # Format reference captions for BLEU scoring
        references = []
        for desc in desc_list:
            # Convert to list of words if it's a string
            if isinstance(desc, str):
                references.append(desc.split())
            else:
                references.append(desc)
        
        # Add to lists for BLEU calculation
        y.append(references)
        yhat.append(pred_caption)
    
    # Calculate BLEU scores
    bleu1 = corpus_bleu(y, yhat, weights=(1.0, 0, 0, 0))
    bleu2 = corpus_bleu(y, yhat, weights=(0.5, 0.5, 0, 0))
    bleu3 = corpus_bleu(y, yhat, weights=(0.3, 0.3, 0.3, 0))
    bleu4 = corpus_bleu(y, yhat, weights=(0.25, 0.25, 0.25, 0.25))
    
    # Print results
    print('BLEU-1: %f' % bleu1)
    print('BLEU-2: %f' % bleu2)
    print('BLEU-3: %f' % bleu3)
    print('BLEU-4: %f' % bleu4)
    
    # Return metrics in a dictionary for easy access
    metrics = {
        'bleu-1': bleu1 * 100,
        'bleu-2': bleu2 * 100,
        'bleu-3': bleu3 * 100,
        'bleu-4': bleu4 * 100
    }
    
    return metrics, yhat

def prepare_evaluation_data(image_dir, captions_file, vocab):
    """
    Prepare all data for evaluation without relying on a separate test list.
    
    Args:
        image_dir: Directory containing images
        captions_file: File containing captions
        vocab: The vocabulary dictionary
    
    Returns:
        all_descriptions: Dictionary mapping image IDs to lists of reference captions
        all_features: Dictionary mapping image IDs to image features
    """
    # Load captions for all images
    all_descriptions = {}
    with open(captions_file, 'r', encoding='utf-8-sig') as f:
        for line in f:
            try:
                tokens = line.strip().split('\t')
                img_name = tokens[0].split('#')[0]
                img_id = img_name.split('.')[0]
                caption = tokens[1]
                
                if img_id not in all_descriptions:
                    all_descriptions[img_id] = []
                
                # Format caption with start and end tokens
                formatted_caption = '<start> ' + caption + ' <end>'
                all_descriptions[img_id].append(formatted_caption)
            except:
                continue
    
    # Load or extract image features
    all_features = {}
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Process all images that have descriptions
    for img_id in tqdm(all_descriptions.keys(), desc="Extracting image features"):
        img_path = os.path.join(image_dir, img_id + '.jpg')
        if os.path.exists(img_path):
            # Load and transform image
            image = Image.open(img_path).convert('RGB')
            image_tensor = transform(image)
            all_features[img_id] = image_tensor
    
    return all_descriptions, all_features

# New function for plotting metrics
def plot_metrics(train_losses, evaluation_metrics, batch_losses=None, output_dir='plots'):
    """
    Plot training losses and evaluation metrics.
    
    Args:
        train_losses: List of training losses per epoch
        evaluation_metrics: Dictionary of evaluation metrics per epoch
        batch_losses: List of lists containing batch losses for each epoch
        output_dir: Directory to save plots
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Set Matplotlib style for better looking plots
    plt.style.use('ggplot')
    
    # Plot training loss
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(train_losses) + 1), train_losses, marker='o', linestyle='-', color='blue')
    plt.title('Training Loss per Epoch', fontsize=16)
    plt.xlabel('Epoch', fontsize=14)
    plt.ylabel('Loss', fontsize=14)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f'{output_dir}/training_loss.png', dpi=300)
    plt.close()
    
    # Plot batch losses if provided (for the last epoch)
    if batch_losses and len(batch_losses) > 0:
        plt.figure(figsize=(12, 6))
        # Plot the most recent epoch's batch losses
        latest_batch_losses = batch_losses[-1]
        plt.plot(range(1, len(latest_batch_losses) + 1), latest_batch_losses, marker='.', linestyle='-', color='green')
        plt.title(f'Batch Losses for Epoch {len(train_losses)}', fontsize=16)
        plt.xlabel('Batch', fontsize=14)
        plt.ylabel('Loss', fontsize=14)
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(f'{output_dir}/latest_batch_losses.png', dpi=300)
        
        # Plot a heatmap of all batch losses across epochs
        if len(batch_losses) > 1:  # Only if we have multiple epochs
            plt.figure(figsize=(12, 8))
            # Pad shorter batches lists to have the same length
            max_batches = max(len(batches) for batches in batch_losses)
            padded_batch_losses = [
                losses + [np.nan] * (max_batches - len(losses)) 
                for losses in batch_losses
            ]
            
            # Create heatmap
            plt.imshow(padded_batch_losses, aspect='auto', cmap='viridis')
            plt.colorbar(label='Loss')
            plt.title('Batch Losses Across Epochs', fontsize=16)
            plt.xlabel('Batch', fontsize=14)
            plt.ylabel('Epoch', fontsize=14)
            plt.tight_layout()
            plt.savefig(f'{output_dir}/batch_losses_heatmap.png', dpi=300)
        
        plt.close()
    
    # Plot evaluation metrics
    if evaluation_metrics:
        # Extract metrics
        epochs = sorted(evaluation_metrics.keys())
        metrics_dict = {}
        
        # Initialize metric dictionaries
        for epoch_idx in epochs:
            for metric_name in evaluation_metrics[epoch_idx].keys():
                if metric_name not in metrics_dict:
                    metrics_dict[metric_name] = []
                metrics_dict[metric_name].append(evaluation_metrics[epoch_idx][metric_name])
        
        # Plot each metric separately
        for metric_name, values in metrics_dict.items():
            plt.figure(figsize=(10, 6))
            plt.plot(epochs, values, marker='o', linestyle='-', color='purple')
            plt.title(f'{metric_name.upper()} Score per Epoch', fontsize=16)
            plt.xlabel('Epoch', fontsize=14)
            plt.ylabel(f'{metric_name.upper()} Score', fontsize=14)
            plt.grid(True)
            plt.tight_layout()
            plt.savefig(f'{output_dir}/{metric_name}_score.png', dpi=300)
            plt.close()
        
        # Plot all BLEU scores together
        plt.figure(figsize=(12, 8))
        colors = ['blue', 'green', 'orange', 'red']
        for i, metric_name in enumerate(['bleu-1', 'bleu-2', 'bleu-3', 'bleu-4']):
            if metric_name in metrics_dict:
                plt.plot(epochs, metrics_dict[metric_name], marker='o', linestyle='-', 
                         label=f'{metric_name.upper()}', color=colors[i % len(colors)])
        
        plt.title('BLEU Scores per Epoch', fontsize=16)
        plt.xlabel('Epoch', fontsize=14)
        plt.ylabel('Score', fontsize=14)
        plt.legend(loc='best')
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(f'{output_dir}/all_bleu_scores.png', dpi=300)
        plt.close()
        
        # Create a comprehensive dashboard plot
        plt.figure(figsize=(15, 10))
        
        # Plot 1: Training Loss
        plt.subplot(2, 2, 1)
        plt.plot(range(1, len(train_losses) + 1), train_losses, marker='o', linestyle='-', color='blue')
        plt.title('Training Loss', fontsize=14)
        plt.xlabel('Epoch', fontsize=12)
        plt.ylabel('Loss', fontsize=12)
        plt.grid(True)
        
        # Plot 2: BLEU-1 and BLEU-2
        plt.subplot(2, 2, 2)
        if 'bleu-1' in metrics_dict and 'bleu-2' in metrics_dict:
            plt.plot(epochs, metrics_dict['bleu-1'], marker='o', linestyle='-', label='BLEU-1', color='green')
            plt.plot(epochs, metrics_dict['bleu-2'], marker='s', linestyle='--', label='BLEU-2', color='purple')
            plt.title('BLEU-1 and BLEU-2 Scores', fontsize=14)
            plt.xlabel('Epoch', fontsize=12)
            plt.ylabel('Score', fontsize=12)
            plt.legend(loc='best')
            plt.grid(True)
        
        # Plot 3: BLEU-3 and BLEU-4
        plt.subplot(2, 2, 3)
        if 'bleu-3' in metrics_dict and 'bleu-4' in metrics_dict:
            plt.plot(epochs, metrics_dict['bleu-3'], marker='o', linestyle='-', label='BLEU-3', color='orange')
            plt.plot(epochs, metrics_dict['bleu-4'], marker='s', linestyle='--', label='BLEU-4', color='red')
            plt.title('BLEU-3 and BLEU-4 Scores', fontsize=14)
            plt.xlabel('Epoch', fontsize=12)
            plt.ylabel('Score', fontsize=12)
            plt.legend(loc='best')
            plt.grid(True)
        
        # Plot 4: Most recent batch losses
        plt.subplot(2, 2, 4)
        if batch_losses and len(batch_losses) > 0:
            latest_batch_losses = batch_losses[-1]
            plt.plot(range(1, len(latest_batch_losses) + 1), latest_batch_losses, marker='.', 
                     linestyle='-', color='green')
            plt.title(f'Batch Losses (Epoch {len(train_losses)})', fontsize=14)
            plt.xlabel('Batch', fontsize=12)
            plt.ylabel('Loss', fontsize=12)
            plt.grid(True)
        
        plt.tight_layout()
        plt.savefig(f'{output_dir}/training_dashboard.png', dpi=300)
        plt.close()
    
    print(f"Plots saved to {output_dir} directory")

# Modified main function to use all data for evaluation with plotting
def main():
    """Main training function with updated evaluation methodology to use all data."""
    try:
        # Configuration
        config = {
            'image_dir': '/kaggle/input/flickr8k/Flickr_Data/Flickr_Data/Images/',
            'captions_file': '/kaggle/input/multi-lingual-flickr8k/italian_caption_p2_v2.txt',
            'batch_size': 16,
            'embed_dim': 384,
            'hidden_dim': 768,
            'num_epochs': 100,
            'learning_rate': 1e-3,
            'train_val_split': 0.4,
            'plots_dir': 'training_plots'  # Directory to save plots
        }
        
        # Create plots directory
        os.makedirs(config['plots_dir'], exist_ok=True)
        
        # Create vocabulary
        vocab = create_vocabulary(config['captions_file'])
        
        # Create dataset for training
        full_dataset = SimpleDataset(
            config['image_dir'],
            config['captions_file'],
            vocab
        )
        
        # Split dataset for training and validation
        train_size = int(config['train_val_split'] * len(full_dataset))
        val_size = len(full_dataset) - train_size
        train_dataset, val_dataset = torch.utils.data.random_split(
            full_dataset, [train_size, val_size]
        )
        
        # Create train dataloader
        train_loader = DataLoader(
            train_dataset,
            batch_size=config['batch_size'],
            shuffle=True,
            num_workers=0
        )
        
        # Prepare all data for final evaluation
        all_descriptions, all_features = prepare_evaluation_data(
            config['image_dir'],
            config['captions_file'],
            vocab
        )
        
        # Initialize model and training components
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {device}")
        
        model = LightweightCaptioningModel(
            vocab_size=len(vocab),
            embed_dim=config['embed_dim'],
            hidden_dim=config['hidden_dim']
        ).to(device)
        
        criterion = nn.CrossEntropyLoss(ignore_index=vocab['<pad>'])
        optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])
        
        # Initialize lists to store metrics for plotting
        train_losses = []
        all_batch_losses = []
        evaluation_metrics = {}
        
        # Training loop with periodic evaluation on validation set
        best_bleu4 = 0
        for epoch in range(config['num_epochs']):
            # Training
            train_loss, batch_losses = train_epoch(
                model, train_loader, criterion, optimizer,
                device, epoch, config['num_epochs']
            )
            
            # Store losses for plotting
            train_losses.append(train_loss)
            all_batch_losses.append(batch_losses)
            
            print(f"\nPerforming full evaluation at epoch {epoch + 1}...")
            metrics, _ = evaluate_model(
                model, 
                all_descriptions, 
                all_features, 
                vocab, 
                30,  # max_length
                device
            )
            
            # Store metrics for plotting
            evaluation_metrics[epoch + 1] = metrics
            
            # Log progress
            print(f'\nEpoch {epoch + 1} Summary:')
            print(f'Training Loss: {train_loss:.4f}')
            
            # Plot metrics after each epoch
            plot_metrics(train_losses, evaluation_metrics, all_batch_losses, config['plots_dir'])
            
            # Save best model
            if metrics['bleu-4'] > best_bleu4:
                best_bleu4 = metrics['bleu-4']
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'vocab': vocab,
                    'config': config,
                    'metrics': metrics,
                    'train_losses': train_losses,
                    'evaluation_metrics': evaluation_metrics
                }, 'best_italian_captioner.pth')
                print(f'Saved new best model with BLEU-4: {best_bleu4:.4f}')
                
        # Final evaluation on the entire dataset
        print("\nPerforming final evaluation on the entire dataset...")
        final_metrics, predictions = evaluate_model(
            model, 
            all_descriptions, 
            all_features, 
            vocab, 
            30,  # max_length
            device
        )
        
        # Add final metrics to evaluation metrics
        evaluation_metrics[config['num_epochs']] = final_metrics
        
        # Generate final plots
        plot_metrics(train_losses, evaluation_metrics, all_batch_losses, config['plots_dir'])
        
        print("\nFinal Evaluation Results:")
        for metric, value in final_metrics.items():
            print(f'{metric}: {value:.4f}')
        
        # Save the final model with all training history
        torch.save({
            'epoch': config['num_epochs'],
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'vocab': vocab,
            'config': config,
            'metrics': final_metrics,
            'train_losses': train_losses,
            'evaluation_metrics': evaluation_metrics
        }, 'final_italian_captioner.pth')
        
        print("Training and evaluation completed successfully")
        
    except Exception as e:
        print(f"Error in training/evaluation: {str(e)}")
        # traceback.print_exc()
        raise


def generate_caption(image_path, checkpoint_path, device='cpu'):
    """Generate caption for a single image using the saved model."""
    checkpoint = torch.load(checkpoint_path, map_location=device)
    vocab = checkpoint['vocab']
    config = checkpoint['config']
    
    model = LightweightCaptioningModel(
        vocab_size=len(vocab),
        embed_dim=config['embed_dim'],
        hidden_dim=config['hidden_dim']
    ).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    image = Image.open(image_path).convert('RGB')
    image = transform(image).unsqueeze(0).to(device)
    
    # Generate caption
    with torch.no_grad():
        hidden = model.image_encoder(image)
        hidden = hidden.squeeze(-1).squeeze(-1)
        hidden = model.image_projection(hidden).unsqueeze(0)
        decoder_input = torch.tensor([[vocab['<start>']]]).to(device)
        generated_caption = []
        for _ in range(30):
            embedded = model.embedding(decoder_input)
            embedded = model.embed_projection(embedded)
            output, hidden = model.decoder(embedded, hidden)
            output = model.output(output)
            predicted = output.argmax(2)
            word_idx = predicted[0].item()
            if word_idx == vocab['<end>']:
                break
            generated_caption.append(word_idx)
            decoder_input = predicted
    
    idx_to_word = {v: k for k, v in vocab.items()}
    caption = ' '.join([idx_to_word.get(idx, '<unk>') for idx in generated_caption])
    return caption

if __name__ == '__main__':
    main()

In [ ]:
generate_caption("/kaggle/input/flickr8k/Flickr_Data/Flickr_Data/Images/1032460886_4a598ed535.jpg","/kaggle/working/best_italian_captioner.pth")

In [ ]:
generate_caption("/kaggle/input/flickr8k/Flickr_Data/Flickr_Data/Images/17273391_55cfc7d3d4.jpg","/kaggle/working/best_italian_captioner.pth")

In [ ]:
generate_caption("/kaggle/input/real-est-2/DSCF3732.JPG","/kaggle/working/best_italian_captioner.pth")

In [ ]:
generate_caption("/kaggle/input/test-real/WIN_20250221_10_11_35_Pro.jpg","/kaggle/working/best_italian_captioner.pth")

In [ ]:
generate_caption("/kaggle/input/real-test-3/DSCF3772.JPG","/kaggle/working/best_italian_captioner.pth")

In [ ]:
generate_caption("/kaggle/input/real-test-3/DSCF3999.JPG","/kaggle/working/best_italian_captioner.pth")

In [ ]:
generate_caption("/kaggle/input/real-test-3/DSCF4140.JPG","/kaggle/working/best_italian_captioner.pth")